In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re
import os

# Verificar archivos disponibles
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mateotorrest/bdua-divipola-2021/Poblacin_Base_de_Datos_nica_de_Afiliados_BDUA_del_rgimen_subsidiado_20251018.csv
/kaggle/input/datasets/mateotorrest/bdua-divipola-2021/DIVIPOLA_CentrosPoblados.csv


In [2]:
# Cargar BDUA 

ruta_bdua = "/kaggle/input/datasets/mateotorrest/bdua-divipola-2021/Poblacin_Base_de_Datos_nica_de_Afiliados_BDUA_del_rgimen_subsidiado_20251018.csv"

df_BDUA = pd.read_csv(ruta_bdua, encoding="latin-1", low_memory=False)

print("Shape:", df_BDUA.shape)
print("\nColumnas:")
for col in df_BDUA.columns.tolist():
    print(f"  - {col}")
print("\nPrimeras 3 filas:")
df_BDUA.head(3)

Shape: (1210054, 14)

Columnas:
  - Genero
  - Grupo etario 
  - CÃ³digo de la entidad
  - Nombre de la entidad
  - RÃ©gimen al que pertenece
  - Tipo de afiliado
  - Estado del afiliado
  - CondiciÃ³n del beneficiario
  - Zona de AfiliaciÃ³n
  - Departamento
  - Municipio
  - Nivel del SisbÃ©n
  - Grupo poblacional del afiliado
  - cantidad

Primeras 3 filas:


,Genero,Grupo etario,CÃ³digo de la entidad,Nombre de la entidad,RÃ©gimen al que pertenece,Tipo de afiliado,Estado del afiliado,CondiciÃ³n del beneficiario,Zona de AfiliaciÃ³n,Departamento,Municipio,Nivel del SisbÃ©n,Grupo poblacional del afiliado,cantidad
0,Masculino,50 a 55,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Rural,CORDOBA,CHINU,2,POBLACIÃN CON SISBEN,6
1,Masculino,19 a 45,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana-Cabecera Municipal,SUCRE,MORROA,N,VÃCTIMAS DEL CONFLICTO ARMADO INTERNO,4
2,Femenino,1 a 5,ESS024,COOSALUD EPS S.A.,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana,MAGDALENA,SABANA DE SAN ANGEL,C,RECIÃN NACIDO Y MENOR DE EDAD DE PADRES NO AF...,1


In [3]:
ruta_bdua = "/kaggle/input/datasets/mateotorrest/bdua-divipola-2021/Poblacin_Base_de_Datos_nica_de_Afiliados_BDUA_del_rgimen_subsidiado_20251018.csv"

df_BDUA = pd.read_csv(ruta_bdua, encoding="utf-8", low_memory=False)

print("Shape:", df_BDUA.shape)
print("\nColumnas:")
for col in df_BDUA.columns.tolist():
    print(f"  - {col}")
print("\nPrimeras 3 filas:")
df_BDUA.head(3)

Shape: (1210054, 14)

Columnas:
  - Genero
  - Grupo etario 
  - Código de la entidad
  - Nombre de la entidad
  - Régimen al que pertenece
  - Tipo de afiliado
  - Estado del afiliado
  - Condición del beneficiario
  - Zona de Afiliación
  - Departamento
  - Municipio
  - Nivel del Sisbén
  - Grupo poblacional del afiliado
  - cantidad

Primeras 3 filas:


,Genero,Grupo etario,Código de la entidad,Nombre de la entidad,Régimen al que pertenece,Tipo de afiliado,Estado del afiliado,Condición del beneficiario,Zona de Afiliación,Departamento,Municipio,Nivel del Sisbén,Grupo poblacional del afiliado,cantidad
0,Masculino,50 a 55,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Rural,CORDOBA,CHINU,2,POBLACIÓN CON SISBEN,6
1,Masculino,19 a 45,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana-Cabecera Municipal,SUCRE,MORROA,N,VÍCTIMAS DEL CONFLICTO ARMADO INTERNO,4
2,Femenino,1 a 5,ESS024,COOSALUD EPS S.A.,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana,MAGDALENA,SABANA DE SAN ANGEL,C,RECIÉN NACIDO Y MENOR DE EDAD DE PADRES NO AFI...,1


In [4]:
#carga DIVIPOLA 

ruta_divipola = "/kaggle/input/datasets/mateotorrest/bdua-divipola-2021/DIVIPOLA_CentrosPoblados.csv"

df_divipola = pd.read_csv(ruta_divipola, encoding="latin-1", sep=";")

# Filtrar solo cabeceras municipales
df_divipola_muni = df_divipola[df_divipola['Tipo'] == 'CM'].copy()

# Quedarnos solo con columnas necesarias
df_divipola_muni = df_divipola_muni[[
    'Código_Departamento', 'Nombre_Departamento',
    'Código_Municipio', 'Nombre_Municipio'
]].rename(columns={
    'Código_Departamento': 'doc_dep',
    'Nombre_Departamento': 'departamento',
    'Código_Municipio':    'cod_muni',
    'Nombre_Municipio':    'municipio'
})

print("Shape DIVIPOLA filtrado:", df_divipola_muni.shape)
df_divipola_muni.head()

Shape DIVIPOLA filtrado: (1104, 4)


,doc_dep,departamento,cod_muni,municipio
0,5,ANTIOQUIA,5001,MEDELLÍN
28,5,ANTIOQUIA,5002,ABEJORRAL
32,5,ANTIOQUIA,5004,ABRIAQUÍ
34,5,ANTIOQUIA,5021,ALEJANDRÍA
35,5,ANTIOQUIA,5030,AMAGÁ


In [5]:
import unicodedata
import re

def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'\s+', ' ', texto)
    return texto

print("✅ Función lista")

✅ Función lista


In [6]:
# Normalizar BDUA
df_BDUA['Departamento'] = df_BDUA['Departamento'].apply(normalizar)
df_BDUA['Municipio']    = df_BDUA['Municipio'].apply(normalizar)

# Normalizar DIVIPOLA
df_divipola_muni['departamento'] = df_divipola_muni['departamento'].apply(normalizar)
df_divipola_muni['municipio']    = df_divipola_muni['municipio'].apply(normalizar)

print("✅ Normalización lista")
print("\nEjemplo BDUA:")
print(df_BDUA[['Departamento', 'Municipio']].head(3))
print("\nEjemplo DIVIPOLA:")
print(df_divipola_muni[['departamento', 'municipio']].head(3))

✅ Normalización lista

Ejemplo BDUA:
  Departamento            Municipio
0      CORDOBA                CHINU
1        SUCRE               MORROA
2    MAGDALENA  SABANA DE SAN ANGEL

Ejemplo DIVIPOLA:
   departamento  municipio
0     ANTIOQUIA   MEDELLIN
28    ANTIOQUIA  ABEJORRAL
32    ANTIOQUIA   ABRIAQUI


In [7]:
# Cruce con Divipola

df_bdua_divipola = df_BDUA.merge(
    df_divipola_muni[['departamento', 'municipio', 'cod_muni', 'doc_dep']],
    left_on  = ['Departamento', 'Municipio'],
    right_on = ['departamento', 'municipio'],
    how      = 'left'
)

# Ver cuántos cruzaron bien
total     = len(df_bdua_divipola)
con_match = df_bdua_divipola['cod_muni'].notna().sum()
sin_match = df_bdua_divipola['cod_muni'].isna().sum()

print(f"Total registros:         {total:,}")
print(f"Con cruce DIVIPOLA:      {con_match:,} ({con_match/total*100:.1f}%)")
print(f"Sin cruce (descartados): {sin_match:,} ({sin_match/total*100:.1f}%)")

Total registros:         1,210,054
Con cruce DIVIPOLA:      1,043,167 (86.2%)
Sin cruce (descartados): 166,887 (13.8%)


In [9]:
# Primero verificamos los nombres exactos de columnas
print("Columnas exactas del dataframe:")
for col in df_bdua_divipola.columns.tolist():
    print(f"  '{col}'")

Columnas exactas del dataframe:
  'Genero'
  'Grupo etario '
  'Código de la entidad'
  'Nombre de la entidad'
  'Régimen al que pertenece'
  'Tipo de afiliado'
  'Estado del afiliado'
  'Condición del beneficiario'
  'Zona de Afiliación'
  'Departamento'
  'Municipio'
  'Nivel del Sisbén'
  'Grupo poblacional del afiliado'
  'cantidad'
  'departamento'
  'municipio'
  'cod_muni'
  'doc_dep'


In [10]:
df_BDUA_limpio = df_bdua_divipola.dropna(subset=['cod_muni']).copy()

# Primero eliminamos espacios extra en nombres de columnas
df_BDUA_limpio.columns = df_BDUA_limpio.columns.str.strip()

# Dejar solo columnas relevantes
cols_finales = [
    'Genero', 'Grupo etario', 'Código de la entidad', 'Nombre de la entidad',
    'Régimen al que pertenece', 'Tipo de afiliado', 'Estado del afiliado',
    'Condición del beneficiario', 'Zona de Afiliación', 'Departamento',
    'Municipio', 'Nivel del Sisbén', 'Grupo poblacional del afiliado',
    'cantidad', 'doc_dep', 'departamento', 'cod_muni', 'municipio'
]
df_BDUA_limpio = df_BDUA_limpio[cols_finales]

print("Shape final:", df_BDUA_limpio.shape)
df_BDUA_limpio.head()

Shape final: (1043167, 18)


,Genero,Grupo etario,Código de la entidad,Nombre de la entidad,Régimen al que pertenece,Tipo de afiliado,Estado del afiliado,Condición del beneficiario,Zona de Afiliación,Departamento,Municipio,Nivel del Sisbén,Grupo poblacional del afiliado,cantidad,doc_dep,departamento,cod_muni,municipio
0,Masculino,50 a 55,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Rural,CORDOBA,CHINU,2,POBLACIÓN CON SISBEN,6,23.0,CORDOBA,23182.0,CHINU
1,Masculino,19 a 45,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana-Cabecera Municipal,SUCRE,MORROA,N,VÍCTIMAS DEL CONFLICTO ARMADO INTERNO,4,70.0,SUCRE,70473.0,MORROA
3,Masculino,45 a 50,ESS024,COOSALUD EPS S.A.,Subsidiado,CABEZA DE FAMILIA,Activo,NO APLICA,Urbana,MAGDALENA,REMOLINO,N,VÍCTIMAS DEL CONFLICTO ARMADO INTERNO,55,47.0,MAGDALENA,47605.0,REMOLINO
4,Femenino,< 1,EPSS41,NUEVA EPS S.A.,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana,META,VILLAVICENCIO,N,RECIÉN NACIDO Y MENOR DE EDAD DE PADRES NO AFI...,7,50.0,META,50001.0,VILLAVICENCIO
6,Femenino,45 a 50,EPSS37,NUEVA EPS S.A. -CM,Subsidiado,CABEZA DE FAMILIA,Activo,NO APLICA,Urbana,ANTIOQUIA,PUERTO NARE,2,POBLACIÓN CON SISBEN,5,5.0,ANTIOQUIA,5585.0,PUERTO NARE


In [11]:
df_BDUA_limpio.to_parquet("/kaggle/working/df_BDUA_limpio.parquet", index=False)

print("✅ Archivo guardado como Parquet")
print(f"Filas: {len(df_BDUA_limpio):,}")
print(f"Columnas: {df_BDUA_limpio.columns.tolist()}")

✅ Archivo guardado como Parquet
Filas: 1,043,167
Columnas: ['Genero', 'Grupo etario', 'Código de la entidad', 'Nombre de la entidad', 'Régimen al que pertenece', 'Tipo de afiliado', 'Estado del afiliado', 'Condición del beneficiario', 'Zona de Afiliación', 'Departamento', 'Municipio', 'Nivel del Sisbén', 'Grupo poblacional del afiliado', 'cantidad', 'doc_dep', 'departamento', 'cod_muni', 'municipio']
